# 02 · 时序 + 接触点坐标 → 方向、投影与空间场（Fig2）

先运行 Notebook 01。这里读取的是 **刚刚从原始 SEEG 计算的事件表**，不读取论文统计表或现成 panel。

Fig2B 的核心问题是：训练时加入空间信息，能否改善独立 recording block 的方向预测？两臂都保留所有训练事件；缺失坐标的事件只缺空间视图。测试时仅用 rank 模板分配标签，空间方向只用于评分，两个方法在同一可评分集合上比较。C 的空间场是模型化描述，不等于连续组织中已测得的传播速度。

In [ ]:
from pathlib import Path
import sys, os, json
# Only insert this independently copied package, never the original repository.
CODE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
import numpy as np
import matplotlib.pyplot as plt
from paper_paths import CODE_ROOT, INPUT_ROOT, OUTPUT_ROOT
from workflows.common import new_session, publish_session, previous_session, write_json
from plotting import computational as plot
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from analysis.from_events import cluster_events, direction_and_crossfit, template_fields, illustrative_grid, spatial_templates
from analysis.src.topic5_interictal_direction_rose import fit_event_directions_3d
SOURCE = previous_session('fig1')
raw_config=json.loads((SOURCE/'run_report.json').read_text())['configuration']
with np.load(SOURCE/'events.npz', allow_pickle=False) as z: events = {k:z[k] for k in z.files}
xyz=events.pop('coords_mm'); clustered=cluster_events(events)
OUT=new_session('fig2'); FIG=OUT/'figures'
print('输入来自本次原始处理:', SOURCE)


**Fig2A/B，方向：** 对每个事件，仅用有坐标的参与通道拟合 `rank = intercept + β·xyz`。rank 越大越晚，所以 `β/||β||` 是 early→late；如果改用 earliness=−z(rank)，则需负梯度。不能再对已经 early→late 的方向翻号。数学不可估事件保留 NaN，不删除其 timing 训练信息。

In [ ]:
directions = fit_event_directions_3d(np.where(events['participating'], events['ranks'], np.nan), xyz)
plot.save(plot.spatial_directions(directions, xyz, clustered), FIG, 'fig2_A_B_rank_gradient', '从新 rank 与三维触点坐标拟合 early-to-late 方向。极坐标图仅将三维方向显示于固定 MRI x/y 平面。')
print('可估方向:', np.isfinite(directions['directions']).all(1).sum()); plt.show()


**Fig2B，Timing 与 Timing+Space：** alternating recording-block 双向交叉验证；空间视图按训练折总方差等权缩放。1000 次 held-out block 内方向置换；50 次 training-space shuffle 重新拟合空间模型。冻结训练后才评分，不能用测试方向选标签。曲线显示本例两个折，不能充当论文 26 人统计。

In [ ]:
directions, gain = direction_and_crossfit(events, xyz, n_null=1000, n_train_null=50)
plot.save(plot.crossfit_scores(gain), FIG, 'fig2_B_crossfit', '重新训练两种模板模型并在相同 held-out 事件上比较。null 由本次 block 内置换产生。')
print('本例信息增益:', gain['spatial_information_gain']); plt.show()


**Fig2C，投影：** 交叉验证评分完成后，另外在全部事件上拟合Timing+Space模板，用于C/D的描述性场；这个全量拟合不回流到B的测试分数。由新模板的 early→late 轴定义纵轴，从剩余空间主方向定义横轴，以同一个 robust length scale 归一化。这里把 TA/TB 都放到 TA 平面，比较时不分别转到各自最有利方向。原实现同时返回 bootstrap/leave-one-shaft-out 资格判断；短样本未通过资格时只能标为说明图。

In [ ]:
clustered=spatial_templates(events,xyz,directions)
write_json(OUT/'full_timing_plus_space_fit.json',clustered['fit'])
field_record = template_fields(events, clustered, xyz, n_boot=200,
    subject_id=raw_config['subject'], subject=raw_config['subject'], dataset=raw_config.get('dataset','raw_demo'))
field_a = illustrative_grid(events, clustered, xyz, 0)
field_b = illustrative_grid(events, clustered, xyz, 1, plane=field_a['plane'])
print('方向与场资格:', field_record['status'], field_record.get('direction_validity'))
print('归一化长度/mm:', field_a['plane']['scale_mm'], '核宽:', field_a['plane']['sigma'])


**Fig2C，场：** `F(x)=Σ K(x,xi)·support_i·earliness_i / Σ K(x,xi)·support_i`，`K=exp(−distance²/(2σ²))`。support 是该模板的真实参与率；σ 由投影后触点间距估计。弱支持区留白；不外推成已验证的组织动力学。

In [ ]:
plot.save(plot.field_panels([field_a,field_b],str(field_record['axis_pair'].get('strict_stability_pass',False))), FIG, 'fig2_C_fields', '由新模板、参与率及固定 TA 平面计算 Gaussian 场。弱支持区留白；资格状态与原输出一同保存。')
plt.show()


**Fig2D：共享平面场的相关。** 计算两模板在同一平面触点处的平滑场 Pearson r；它不同于两个rank-gradient轴的余弦。打乱TB的触点身份（earliness和support一起移动），重新平滑形成null。队列层每患者一行，不能把触点当患者。

In [ ]:
from analysis.from_events import shared_field_similarity
similarity=shared_field_similarity(field_record,n_perm=1000)
print('共享场统计:', {k:v for k,v in similarity.items() if k!='channel_null'})
if similarity['status']=='OK':
    fig,ax=plt.subplots(figsize=(6,4),layout='constrained')
    ax.hist(similarity['channel_null'],bins=25,color='gray')
    ax.axvline(similarity['observed_shared_field_r'],color='red',label='observed shared-field r')
    ax.legend(); ax.set(xlabel='TA/TB shared-field Pearson r',title='Fig2D: one-patient full-contact shuffle')
    plot.save(fig,FIG,'fig2_D_shared_field_null','共同平面触点处的TA/TB场相关及全触点置换null。该量不同于方向轴余弦，当前仅一个患者。');plt.show()
write_json(OUT/'shared_field_similarity.json',similarity)
cosine=float(field_a['direction']@field_b['direction'])
print('本例 TA/TB 有符号方向余弦:', cosine)
write_json(OUT/'crossfit.json',gain); write_json(OUT/'field_record.json',field_record)
np.savez_compressed(OUT/'spatial_fields.npz', X_a=field_a['grid']['X'],Y_a=field_a['grid']['Y'],
                    F_a=field_a['field'][0],S_a=field_a['field'][1],X_b=field_b['grid']['X'],
                    Y_b=field_b['grid']['Y'],F_b=field_b['field'][0],S_b=field_b['field'][1])
publish_session('fig2',OUT,dict(input_events=str(SOURCE/'events.npz'),n_events=events['ranks'].shape[1],
               template_method=clustered['fit']['method'],template_policy=clustered['fit']['event_policy'],cosine=cosine,spatial_gain=gain['spatial_information_gain'],field_status=field_record['status']))
print(OUT)


**后续 Fig3 / Fig5 临床能量场接口。** `analysis/spectral_fields.py::patient_energy_analysis` 接受真实参考后 SEEG、采样率、通道名和 t_spectral 时间原点，计算频带功率、远端 −120 到 −90 s robust-z、严格名称对齐以及冻结 interictal 场的评分。需要真实基线与 onset 元数据；不能用 SNN 代替临床能量场。